# Azure Databricks for Data Engineering - Master Fundamentals

Welcome to **Project 14: Azure Databricks for Data Engineering** in the *Data Engineering Diaries* series by Sathiya Murthy.

### Overview
This notebook provides hands-on execution and architectural walkthroughs for cloud-native big data engineering on **Microsoft Azure Databricks**.

---

### Key Capabilities Demonstrated
1. **ADLS Gen2 Storage Integration**: Service Principal (OAuth 2.0) and ABFSS direct access protocols.
2. **Azure Key Vault Integration**: Databricks secret scopes and safe credential management without plaintext leaks.
3. **Azure Unity Catalog**: Centralized account-level governance, storage credentials, and external locations.
4. **End-to-End Medallion ETL Pipeline**: Bronze (Raw landing) → Silver (Cleaned/Deduplicated) → Gold (Aggregated) → Azure Synapse DW.
5. **Azure Data Factory (ADF) Orchestration**: Parameter passing, widget extraction, and exit payloads.
6. **Cost Optimization & Monitoring**: Cluster policies, Spot VM savings, and Log Analytics telemetry.

In [ ]:
import os
import sys
import logging

# Ensure project modules are importable
sys.path.append("..")

from src.adls_integration import ADLSGen2Manager
from src.azure_key_vault import AzureKeyVaultManager
from src.unity_catalog_azure import UnityCatalogAzureManager
from src.azure_etl_pipeline import AzureETLPipeline, AzureETLPipelineConfig
from src.adf_databricks import ADFDatabricksHandler
from src.monitoring_azure import AzureMonitorManager
from src.cost_optimization import CostOptimizationManager
from src.security_patterns import AzureSecurityPatternsManager
from src.devops_integration import DevOpsIntegrationManager

print("✅ Azure Databricks Data Engineering modules imported successfully.")

## 1. ADLS Gen2 Storage Integration & ABFSS Paths

Azure Data Lake Storage Gen2 (ADLS Gen2) is accessed using the high-performance **ABFS (Azure Blob File System) driver**:
`abfss://<container>@<storage_account>.dfs.core.windows.net/<path>`

Authentication patterns:
- **Service Principal (OAuth 2.0)**: Client ID + Client Secret + Tenant ID
- **Azure Managed Identity (MSI)**: Recommended for Unity Catalog
- **Storage Account Access Keys**: Legacy pattern (avoid in prod)

In [ ]:
adls_mgr = ADLSGen2Manager(
    storage_account_name="stmedallionprod",
    client_id="mock-client-id-12345",
    client_secret="mock-client-secret-67890",
    tenant_id="mock-tenant-id-aaaaa"
)

# Generate ABFSS URIs for Medallion Containers
bronze_uri = adls_mgr.get_abfss_path("bronze", "transactions/2026")
silver_uri = adls_mgr.get_abfss_path("silver", "tables/dim_customer")
gold_uri = adls_mgr.get_abfss_path("gold", "tables/fact_daily_sales")

print("Bronze Path: ", bronze_uri)
print("Silver Path: ", silver_uri)
print("Gold Path:   ", gold_uri)

# Retrieve Spark Hadoop Configs for Service Principal Auth
spark_conf = adls_mgr.get_spark_service_principal_configs()
for k, v in list(spark_conf.items())[:2]:
    print(f"Spark Config: {k} = {v}")

## 2. Azure Key Vault Backed Secret Scopes

Databricks Secret Scopes back credentials with Azure Key Vault (AKV).
Retrieval syntax in notebooks:
`dbutils.secrets.get(scope="akv-prod-scope", key="synapse-sql-auth")`

Benefits:
- Automatic log redaction `[REDACTED]`
- Central secret rotation without editing Spark jobs or code
- Full Azure Active Directory audit trail in Azure Monitor

In [ ]:
kv_mgr = AzureKeyVaultManager(scope_name="akv-prod-scope")

# Demonstrate safe secret retrieval (mocked when outside active Databricks cluster)
secret = kv_mgr.get_secret("synapse-sql-auth", dbutils=None)
print("Retrieved Secret Scope Key: synapse-sql-auth")
print("Secret Value (masked for logs):", "*" * len(secret))

## 3. Azure Unity Catalog External Locations & Storage Credentials

Unity Catalog unifies governance across Azure Databricks workspaces.
- **Storage Credentials**: Encapsulate Azure Access Connector Managed Identity.
- **External Locations**: Bind ADLS Gen2 path (`abfss://...`) to Storage Credential.
- **Three-Level Namespace**: `catalog.schema.table`

In [ ]:
uc_mgr = UnityCatalogAzureManager(catalog_name="prod_catalog")

# Generate External Location DDL
sql_loc = uc_mgr.generate_create_external_location_sql(
    location_name="ext_silver_orders",
    abfss_url="abfss://silver@stmedallionprod.dfs.core.windows.net/orders",
    credential_name="access_connector_mi_cred"
)

# Generate Access Grant SQL
sql_grant = uc_mgr.generate_grant_access_sql(
    principal="data_analysts_sg",
    privilege="SELECT",
    object_type="TABLE",
    object_name="prod_catalog.gold.fact_daily_sales"
)

print("Generated DDL 1:\n", sql_loc)
print("\nGenerated DDL 2:\n", sql_grant)

## 4. End-to-End Medallion Pipeline Execution & Synapse Export

Executes full Medallion workflow:
1. **Bronze**: Raw JSON read from ADLS Gen2 with schema validation & metadata injection.
2. **Silver**: Data quality filtering, timestamp parsing, deduplication, and MERGE INTO Delta table.
3. **Gold**: Business KPI aggregation (revenue, daily totals).
4. **Synapse Export**: Staging to ADLS Gen2 & bulk copy into Azure Synapse Dedicated SQL Pool.

In [ ]:
etl_pipeline = AzureETLPipeline()
pipeline_metrics = etl_pipeline.run_pipeline()

print("ETL Execution Metrics:")
for k, v in pipeline_metrics.items():
    print(f"  {k}: {v}")

## 5. ADF Integration, Cost Optimization & Azure Monitoring

- **ADF Handler**: Extract widgets and send JSON exit status back to ADF.
- **Cost Optimization**: Calculate savings from Spot VM workers and enforce cluster policies.
- **Azure Monitor**: Stream structured telemetry to Log Analytics workspace.

In [ ]:
# 1. ADF Parameter & Exit Handling
adf = ADFDatabricksHandler(dbutils=None)
params = adf.extract_adf_parameters()
exit_payload = adf.exit_notebook_with_status("SUCCESS", 100000, metrics={"duration": 15.2})
print("ADF Exit Payload:", exit_payload)

# 2. Cost Estimate Calculation
cost_mgr = CostOptimizationManager()
estimate = cost_mgr.estimate_cluster_cost("Standard_D4ds_v5", num_workers=4, duration_hours=8.0, use_spot_workers=True)
print(f"Cluster Savings vs On-Demand: {estimate['estimated_savings_percent']}% (${estimate['total_estimated_cost_usd']} total)")

# 3. Telemetry Event Logging
monitor = AzureMonitorManager(env="production")
monitor.log_pipeline_event("NotebookExecution", "GoldStep", "SUCCESS", "Notebook execution finished successfully.")